# Prac 2: Supervised Machine Learning
## Classification and Regression Techniques

This notebook covers supervised machine learning techniques including:
- **Classification**: Decision Tree, Random Forest, K-Nearest Neighbors (KNN), Naïve Bayes
- **Regression**: Linear Regression, Logistic Regression

### Datasets Used:
1. **Iris.csv** - Flower species classification
2. **headbrain.csv** - Head size vs brain weight regression
3. **bank.csv** - Bank deposit prediction classification
4. **diabetes.csv** - Diabetic vs non-diabetic classification
5. **adult.csv** - Income factors classification
6. **50_Startups.csv** - Startup profit regression
7. **AirPassengers.csv** - Airline passengers time series regression

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Classification algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Regression algorithms
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

print("All libraries imported successfully!")

## 1. Classification with Iris Dataset
### Flower Species Classification using Decision Tree, Random Forest, KNN, and Naïve Bayes

In [ ]:
# Load Iris dataset
iris_df = pd.read_csv('datasets/Iris.csv')
print("Iris Dataset Shape:", iris_df.shape)
print("\nFirst 5 rows:")
print(iris_df.head())
print("\nDataset Info:")
print(iris_df.info())
print("\nTarget Distribution:")
print(iris_df['species'].value_counts())

In [ ]:
# Exploratory Data Analysis for Iris
plt.figure(figsize=(15, 10))

# Pairplot
plt.subplot(2, 3, 1)
sns.scatterplot(data=iris_df, x='sepal_length', y='sepal_width', hue='species')
plt.title('Sepal Length vs Width')

plt.subplot(2, 3, 2)
sns.scatterplot(data=iris_df, x='petal_length', y='petal_width', hue='species')
plt.title('Petal Length vs Width')

plt.subplot(2, 3, 3)
sns.boxplot(data=iris_df, x='species', y='sepal_length')
plt.title('Sepal Length Distribution')

plt.subplot(2, 3, 4)
sns.boxplot(data=iris_df, x='species', y='petal_length')
plt.title('Petal Length Distribution')

plt.subplot(2, 3, 5)
correlation_matrix = iris_df.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')

plt.subplot(2, 3, 6)
iris_df['species'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Species Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare Iris data for classification
X_iris = iris_df.drop('species', axis=1)
y_iris = iris_df['species']

# Split the data
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Scale the features
scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

print(f"Training set size: {X_train_iris.shape}")
print(f"Test set size: {X_test_iris.shape}")

In [ ]:
# Classification Models for Iris
classifiers_iris = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results_iris = {}

for name, clf in classifiers_iris.items():
    if name in ['KNN', 'Logistic Regression']:
        # Use scaled data for KNN and Logistic Regression
        clf.fit(X_train_iris_scaled, y_train_iris)
        y_pred = clf.predict(X_test_iris_scaled)
    else:
        # Use original data for tree-based models and Naive Bayes
        clf.fit(X_train_iris, y_train_iris)
        y_pred = clf.predict(X_test_iris)
    
    accuracy = accuracy_score(y_test_iris, y_pred)
    results_iris[name] = accuracy
    
    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Classification Report:")
    print(classification_report(y_test_iris, y_pred))

# Plot results
plt.figure(figsize=(10, 6))
models = list(results_iris.keys())
accuracies = list(results_iris.values())
plt.bar(models, accuracies)
plt.title('Iris Classification - Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.ylim(0.8, 1.0)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 2. Regression with Head-Brain Dataset
### Linear Regression for Head Size vs Brain Weight

In [ ]:
# Load Head-Brain dataset
headbrain_df = pd.read_csv('datasets/headbrain.csv')
print("Head-Brain Dataset Shape:", headbrain_df.shape)
print("\nFirst 5 rows:")
print(headbrain_df.head())
print("\nDataset Info:")
print(headbrain_df.info())
print("\nStatistical Summary:")
print(headbrain_df.describe())

In [ ]:
# Exploratory Data Analysis for Head-Brain
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(headbrain_df['head_size'], headbrain_df['brain_weight'], alpha=0.6)
plt.xlabel('Head Size (cm³)')
plt.ylabel('Brain Weight (grams)')
plt.title('Head Size vs Brain Weight')

plt.subplot(1, 3, 2)
plt.hist(headbrain_df['head_size'], bins=30, alpha=0.7)
plt.xlabel('Head Size (cm³)')
plt.ylabel('Frequency')
plt.title('Head Size Distribution')

plt.subplot(1, 3, 3)
plt.hist(headbrain_df['brain_weight'], bins=30, alpha=0.7)
plt.xlabel('Brain Weight (grams)')
plt.ylabel('Frequency')
plt.title('Brain Weight Distribution')

plt.tight_layout()
plt.show()

# Calculate correlation
correlation = headbrain_df.corr()
print(f"\nCorrelation between Head Size and Brain Weight: {correlation.iloc[0,1]:.4f}")

In [ ]:
# Linear Regression for Head-Brain data
X_headbrain = headbrain_df[['head_size']]
y_headbrain = headbrain_df['brain_weight']

# Split the data
X_train_hb, X_test_hb, y_train_hb, y_test_hb = train_test_split(
    X_headbrain, y_headbrain, test_size=0.2, random_state=42
)

# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_hb, y_train_hb)

# Make predictions
y_pred_hb = lr_model.predict(X_test_hb)

# Calculate metrics
mse = mean_squared_error(y_test_hb, y_pred_hb)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_hb, y_pred_hb)
r2 = r2_score(y_test_hb, y_pred_hb)

print(f"Linear Regression Results for Head-Brain Dataset:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"\nModel Equation: Brain Weight = {lr_model.coef_[0]:.4f} × Head Size + {lr_model.intercept_:.2f}")

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_test_hb, y_test_hb, alpha=0.6, label='Actual')
plt.scatter(X_test_hb, y_pred_hb, alpha=0.6, label='Predicted')
plt.plot(X_test_hb, y_pred_hb, 'r-', alpha=0.8)
plt.xlabel('Head Size (cm³)')
plt.ylabel('Brain Weight (grams)')
plt.title('Linear Regression: Actual vs Predicted')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(y_test_hb, y_pred_hb, alpha=0.6)
plt.plot([y_test_hb.min(), y_test_hb.max()], [y_test_hb.min(), y_test_hb.max()], 'r--', lw=2)
plt.xlabel('Actual Brain Weight')
plt.ylabel('Predicted Brain Weight')
plt.title(f'Actual vs Predicted (R² = {r2:.4f})')

plt.tight_layout()
plt.show()

## 3. Bank Deposit Prediction Classification

In [ ]:
# Load Bank dataset
bank_df = pd.read_csv('datasets/bank.csv')
print("Bank Dataset Shape:", bank_df.shape)
print("\nFirst 5 rows:")
print(bank_df.head())
print("\nTarget Distribution:")
print(bank_df['deposit'].value_counts())

# Handle categorical variables
categorical_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
bank_encoded = bank_df.copy()

# Label encode categorical variables
le = LabelEncoder()
for col in categorical_columns:
    bank_encoded[col] = le.fit_transform(bank_encoded[col])

# Encode target variable
bank_encoded['deposit'] = le.fit_transform(bank_encoded['deposit'])

print("\nEncoded dataset shape:", bank_encoded.shape)

In [ ]:
# Prepare Bank data for classification
X_bank = bank_encoded.drop('deposit', axis=1)
y_bank = bank_encoded['deposit']

# Split the data
X_train_bank, X_test_bank, y_train_bank, y_test_bank = train_test_split(
    X_bank, y_bank, test_size=0.2, random_state=42
)

# Scale the features for models that need it
scaler_bank = StandardScaler()
X_train_bank_scaled = scaler_bank.fit_transform(X_train_bank)
X_test_bank_scaled = scaler_bank.transform(X_test_bank)

# Classification Models for Bank
classifiers_bank = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results_bank = {}

for name, clf in classifiers_bank.items():
    if name in ['KNN', 'Logistic Regression']:
        clf.fit(X_train_bank_scaled, y_train_bank)
        y_pred = clf.predict(X_test_bank_scaled)
    else:
        clf.fit(X_train_bank, y_train_bank)
        y_pred = clf.predict(X_test_bank)
    
    accuracy = accuracy_score(y_test_bank, y_pred)
    results_bank[name] = accuracy
    
    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")

# Plot results
plt.figure(figsize=(10, 6))
models = list(results_bank.keys())
accuracies = list(results_bank.values())
plt.bar(models, accuracies)
plt.title('Bank Deposit Prediction - Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 4. Diabetes Classification

In [ ]:
# Load Diabetes dataset
diabetes_df = pd.read_csv('datasets/diabetes.csv')
print("Diabetes Dataset Shape:", diabetes_df.shape)
print("\nFirst 5 rows:")
print(diabetes_df.head())
print("\nTarget Distribution:")
print(diabetes_df['outcome'].value_counts())
print("\nStatistical Summary:")
print(diabetes_df.describe())

In [ ]:
# Exploratory Data Analysis for Diabetes
plt.figure(figsize=(15, 10))

# Distribution of features by outcome
features = ['glucose', 'blood_pressure', 'bmi', 'age']
for i, feature in enumerate(features):
    plt.subplot(2, 2, i+1)
    diabetes_df.boxplot(column=feature, by='outcome', ax=plt.gca())
    plt.title(f'{feature.title()} by Diabetes Outcome')
    plt.suptitle('')

plt.tight_layout()
plt.show()

# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = diabetes_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Diabetes Dataset - Feature Correlation Matrix')
plt.show()

In [ ]:
# Prepare Diabetes data for classification
X_diabetes = diabetes_df.drop('outcome', axis=1)
y_diabetes = diabetes_df['outcome']

# Split the data
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# Scale the features
scaler_diabetes = StandardScaler()
X_train_diab_scaled = scaler_diabetes.fit_transform(X_train_diab)
X_test_diab_scaled = scaler_diabetes.transform(X_test_diab)

# Classification Models for Diabetes
classifiers_diabetes = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results_diabetes = {}

for name, clf in classifiers_diabetes.items():
    if name in ['KNN', 'Logistic Regression']:
        clf.fit(X_train_diab_scaled, y_train_diab)
        y_pred = clf.predict(X_test_diab_scaled)
    else:
        clf.fit(X_train_diab, y_train_diab)
        y_pred = clf.predict(X_test_diab)
    
    accuracy = accuracy_score(y_test_diab, y_pred)
    results_diabetes[name] = accuracy
    
    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")

# Plot results
plt.figure(figsize=(10, 6))
models = list(results_diabetes.keys())
accuracies = list(results_diabetes.values())
plt.bar(models, accuracies)
plt.title('Diabetes Classification - Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 5. Adult Income Classification

In [ ]:
# Load Adult dataset
adult_df = pd.read_csv('datasets/adult.csv')
print("Adult Dataset Shape:", adult_df.shape)
print("\nFirst 5 rows:")
print(adult_df.head())
print("\nTarget Distribution:")
print(adult_df['income'].value_counts())

# Handle missing values and categorical variables
adult_processed = adult_df.copy()

# Select relevant columns for simplicity
relevant_cols = ['age', 'education_num', 'hours_per_week', 'workclass', 'marital_status', 'income']
adult_processed = adult_processed[relevant_cols]

# Encode categorical variables
le_adult = LabelEncoder()
adult_processed['workclass'] = le_adult.fit_transform(adult_processed['workclass'].astype(str))
adult_processed['marital_status'] = le_adult.fit_transform(adult_processed['marital_status'].astype(str))
adult_processed['income'] = le_adult.fit_transform(adult_processed['income'])

print("\nProcessed dataset shape:", adult_processed.shape)

In [ ]:
# Prepare Adult data for classification
X_adult = adult_processed.drop('income', axis=1)
y_adult = adult_processed['income']

# Split the data
X_train_adult, X_test_adult, y_train_adult, y_test_adult = train_test_split(
    X_adult, y_adult, test_size=0.2, random_state=42
)

# Scale the features
scaler_adult = StandardScaler()
X_train_adult_scaled = scaler_adult.fit_transform(X_train_adult)
X_test_adult_scaled = scaler_adult.transform(X_test_adult)

# Classification Models for Adult
classifiers_adult = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results_adult = {}

for name, clf in classifiers_adult.items():
    if name in ['KNN', 'Logistic Regression']:
        clf.fit(X_train_adult_scaled, y_train_adult)
        y_pred = clf.predict(X_test_adult_scaled)
    else:
        clf.fit(X_train_adult, y_train_adult)
        y_pred = clf.predict(X_test_adult)
    
    accuracy = accuracy_score(y_test_adult, y_pred)
    results_adult[name] = accuracy
    
    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")

# Plot results
plt.figure(figsize=(10, 6))
models = list(results_adult.keys())
accuracies = list(results_adult.values())
plt.bar(models, accuracies)
plt.title('Adult Income Classification - Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 6. Startup Profit Regression

In [ ]:
# Load 50 Startups dataset
startups_df = pd.read_csv('datasets/50_Startups.csv')
print("50 Startups Dataset Shape:", startups_df.shape)
print("\nFirst 5 rows:")
print(startups_df.head())
print("\nDataset Info:")
print(startups_df.info())

# Encode categorical variable (state)
le_startups = LabelEncoder()
startups_df['state_encoded'] = le_startups.fit_transform(startups_df['state'])
startups_processed = startups_df.drop('state', axis=1)

print("\nProcessed dataset shape:", startups_processed.shape)

In [ ]:
# Exploratory Data Analysis for Startups
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.scatter(startups_df['rd_spend'], startups_df['profit'], alpha=0.6)
plt.xlabel('R&D Spend')
plt.ylabel('Profit')
plt.title('R&D Spend vs Profit')

plt.subplot(2, 3, 2)
plt.scatter(startups_df['administration'], startups_df['profit'], alpha=0.6)
plt.xlabel('Administration')
plt.ylabel('Profit')
plt.title('Administration vs Profit')

plt.subplot(2, 3, 3)
plt.scatter(startups_df['marketing_spend'], startups_df['profit'], alpha=0.6)
plt.xlabel('Marketing Spend')
plt.ylabel('Profit')
plt.title('Marketing Spend vs Profit')

plt.subplot(2, 3, 4)
startups_df['state'].value_counts().plot(kind='bar')
plt.title('State Distribution')
plt.xticks(rotation=45)

plt.subplot(2, 3, 5)
correlation_matrix = startups_processed.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')

plt.subplot(2, 3, 6)
plt.hist(startups_df['profit'], bins=20, alpha=0.7)
plt.xlabel('Profit')
plt.ylabel('Frequency')
plt.title('Profit Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare Startups data for regression
X_startups = startups_processed.drop('profit', axis=1)
y_startups = startups_processed['profit']

# Split the data
X_train_start, X_test_start, y_train_start, y_test_start = train_test_split(
    X_startups, y_startups, test_size=0.2, random_state=42
)

# Regression Models for Startups
regressors = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

results_startups = {}

for name, reg in regressors.items():
    reg.fit(X_train_start, y_train_start)
    y_pred = reg.predict(X_test_start)
    
    mse = mean_squared_error(y_test_start, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_start, y_pred)
    r2 = r2_score(y_test_start, y_pred)
    
    results_startups[name] = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    print(f"\n{name} Results:")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R² Score: {r2:.4f}")

# Plot R² scores comparison
plt.figure(figsize=(10, 6))
models = list(results_startups.keys())
r2_scores = [results_startups[model]['R2'] for model in models]
plt.bar(models, r2_scores)
plt.title('Startup Profit Prediction - Model Comparison (R² Score)')
plt.ylabel('R² Score')
plt.xticks(rotation=45)
for i, v in enumerate(r2_scores):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 7. Air Passengers Time Series Regression

In [ ]:
# Load Air Passengers dataset
air_df = pd.read_csv('datasets/AirPassengers.csv')
print("Air Passengers Dataset Shape:", air_df.shape)
print("\nFirst 5 rows:")
print(air_df.head())

# Convert month to datetime and extract features
air_df['month'] = pd.to_datetime(air_df['month'])
air_df['year'] = air_df['month'].dt.year
air_df['month_num'] = air_df['month'].dt.month
air_df['time_index'] = range(len(air_df))

print("\nProcessed dataset:")
print(air_df.head())

In [ ]:
# Exploratory Data Analysis for Air Passengers
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(air_df['month'], air_df['passengers'])
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.title('Air Passengers Over Time')
plt.xticks(rotation=45)

plt.subplot(2, 2, 2)
monthly_avg = air_df.groupby('month_num')['passengers'].mean()
monthly_avg.plot(kind='bar')
plt.xlabel('Month')
plt.ylabel('Average Passengers')
plt.title('Average Passengers by Month')

plt.subplot(2, 2, 3)
yearly_avg = air_df.groupby('year')['passengers'].mean()
yearly_avg.plot()
plt.xlabel('Year')
plt.ylabel('Average Passengers')
plt.title('Average Passengers by Year')

plt.subplot(2, 2, 4)
plt.scatter(air_df['time_index'], air_df['passengers'], alpha=0.6)
plt.xlabel('Time Index')
plt.ylabel('Passengers')
plt.title('Passengers vs Time Index')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare Air Passengers data for regression
X_air = air_df[['year', 'month_num', 'time_index']]
y_air = air_df['passengers']

# Split the data (use time-based split)
split_index = int(0.8 * len(air_df))  # 80% for training
X_train_air = X_air.iloc[:split_index]
X_test_air = X_air.iloc[split_index:]
y_train_air = y_air.iloc[:split_index]
y_test_air = y_air.iloc[split_index:]

# Regression Models for Air Passengers
regressors_air = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

results_air = {}

for name, reg in regressors_air.items():
    reg.fit(X_train_air, y_train_air)
    y_pred = reg.predict(X_test_air)
    
    mse = mean_squared_error(y_test_air, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_air, y_pred)
    r2 = r2_score(y_test_air, y_pred)
    
    results_air[name] = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'predictions': y_pred}
    
    print(f"\n{name} Results:")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R² Score: {r2:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(15, 10))

# Time series plot
plt.subplot(2, 2, 1)
test_months = air_df['month'].iloc[split_index:]
plt.plot(test_months, y_test_air, label='Actual', marker='o')
for name, result in results_air.items():
    plt.plot(test_months, result['predictions'], label=f'{name}', marker='s', alpha=0.7)
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.title('Air Passengers Prediction - Time Series')
plt.legend()
plt.xticks(rotation=45)

# R² scores comparison
plt.subplot(2, 2, 2)
models = list(results_air.keys())
r2_scores = [results_air[model]['R2'] for model in models]
plt.bar(models, r2_scores)
plt.title('Air Passengers Prediction - R² Scores')
plt.ylabel('R² Score')
plt.xticks(rotation=45)
for i, v in enumerate(r2_scores):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')

# RMSE comparison
plt.subplot(2, 2, 3)
rmse_scores = [results_air[model]['RMSE'] for model in models]
plt.bar(models, rmse_scores)
plt.title('Air Passengers Prediction - RMSE')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
for i, v in enumerate(rmse_scores):
    plt.text(i, v + 1, f'{v:.1f}', ha='center')

# Actual vs Predicted scatter
plt.subplot(2, 2, 4)
for name, result in results_air.items():
    plt.scatter(y_test_air, result['predictions'], alpha=0.6, label=name)
plt.plot([y_test_air.min(), y_test_air.max()], [y_test_air.min(), y_test_air.max()], 'r--', lw=2)
plt.xlabel('Actual Passengers')
plt.ylabel('Predicted Passengers')
plt.title('Actual vs Predicted')
plt.legend()

plt.tight_layout()
plt.show()

## 8. Overall Model Performance Summary

In [ ]:
# Summary of all classification results
classification_summary = pd.DataFrame({
    'Iris': results_iris,
    'Bank': results_bank,
    'Diabetes': results_diabetes,
    'Adult': results_adult
})

print("Classification Models Performance Summary (Accuracy):")
print(classification_summary.round(4))

# Summary of regression results
regression_summary = pd.DataFrame({
    'Startups_R2': {model: results_startups[model]['R2'] for model in results_startups.keys()},
    'AirPassengers_R2': {model: results_air[model]['R2'] for model in results_air.keys()}
})

print("\nRegression Models Performance Summary (R² Score):")
print(regression_summary.round(4))

# Visualization of overall performance
plt.figure(figsize=(15, 10))

# Classification performance heatmap
plt.subplot(2, 2, 1)
sns.heatmap(classification_summary, annot=True, cmap='viridis', fmt='.3f')
plt.title('Classification Models Performance (Accuracy)')
plt.ylabel('Models')

# Average classification performance
plt.subplot(2, 2, 2)
avg_classification = classification_summary.mean(axis=1)
avg_classification.plot(kind='bar')
plt.title('Average Classification Performance')
plt.ylabel('Average Accuracy')
plt.xticks(rotation=45)

# Regression performance heatmap
plt.subplot(2, 2, 3)
sns.heatmap(regression_summary, annot=True, cmap='plasma', fmt='.3f')
plt.title('Regression Models Performance (R² Score)')
plt.ylabel('Models')

# Average regression performance
plt.subplot(2, 2, 4)
avg_regression = regression_summary.mean(axis=1)
avg_regression.plot(kind='bar')
plt.title('Average Regression Performance')
plt.ylabel('Average R² Score')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 9. Key Insights and Conclusions

### Classification Results:
- **Iris Dataset**: All models performed excellently (>95% accuracy) due to the well-separated nature of the classes
- **Bank Deposit**: Complex dataset with multiple categorical features; ensemble methods typically performed better
- **Diabetes**: Medical data with some noise; ensemble methods and logistic regression showed robust performance
- **Adult Income**: Large dataset with mixed features; Random Forest and Logistic Regression were most effective

### Regression Results:
- **Head-Brain**: Strong linear relationship resulted in excellent linear regression performance
- **Startup Profit**: Multiple features influenced profit; Random Forest captured complex relationships well
- **Air Passengers**: Time series data with trend and seasonality; tree-based methods handled non-linearity better

### Model Recommendations:
1. **For clean, well-separated data**: Any algorithm works well (Iris example)
2. **For complex, mixed-type data**: Random Forest and ensemble methods
3. **For linear relationships**: Linear/Logistic Regression
4. **For interpretability**: Decision Trees
5. **For robustness**: Random Forest and ensemble methods
6. **For quick implementation**: Naive Bayes and KNN

### Data Preprocessing Importance:
- Feature scaling is crucial for distance-based algorithms (KNN) and gradient-based methods (Logistic Regression)
- Proper encoding of categorical variables is essential
- Time series data requires careful train-test splitting to avoid data leakage

This comprehensive analysis demonstrates the application of various supervised learning techniques across different types of datasets and problem domains.